## Lab 8: Train and Quantize Model for TinyML
*Suggested time: ~ 30 minutes*

Accelerator : CPU

The purpose of this lab is to explain by way of examplar code, the following:

- Train a fully connected regression model that predicts Fibonacci series
- Save floating poing and quantized model in Tflite format
- Convert saved model to byte array file (.cc)
- Download this model for use on Microcontroller

### Part A : **emlearn**

Machine learning and fully connected models for microcontrollers using C99 compiler

##### Step 1:

- Create a classical machine learning model using decision tree
- Convert the model into a header (.h) file
- 

In [3]:
# 1. Install emlearn
!pip install -q emlearn scikit-learn numpy

import emlearn
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# 2. Create and train a simple model (AND Gate example)
X = np.array([[0, 0], [255, 0], [0, 255], [255, 255]], dtype=np.int16)
y = np.array([0, 0, 0, 1])

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, y)

# 3. Convert to C header
# The 'inline' method is fastest and requires the least RAM
cmodel = emlearn.convert(model, method='inline')
cmodel.save(file='my_model.h', name='my_model')

# 4. Find where emlearn headers are located so you can download them
print(f"Download the core headers from: {emlearn.includedir}")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Download the core headers from: /home/user/.local/lib/python3.12/site-packages/emlearn


##### Step 2: 

- Create a C based inference code to use the model stored in the header file
- Prepare  input_feature to be be used by the model 
- Call **model_name_predict** function to execute the model and store the value in variable result
- Print the inference output from the model

In [4]:
%%writefile my_emlearn_model.c

#include <stdio.h>
#include <stdint.h>

// 1. Include the model generated in Colab
#include "my_model.h"

int main() {
    printf("--- emlearn Pure C Inference ---\n");

    // 2. Prepare input features (matching the training data range)
    int16_t input_features[2] = {255, 255};

    // 3. Call the generated prediction function
    // Format: [name]_predict(features_array, num_features)
    int32_t result = my_model_predict(input_features, 2);

    printf("Input: [255, 255] -> AI Prediction: %d\n", result);

    return 0;
}

Overwriting my_emlearn_model.c


##### Step 3:
- Compile the model by including the header files of emlearn
- The header files of emlearn are obtained by printing *emlearn.includedir*

In [7]:
!gcc -o ml_model my_emlearn_model.c -I $(python -c "import emlearn; print(emlearn.includedir)")

In [8]:
!./ml_model

--- emlearn Pure C Inference ---
Input: [255, 255] -> AI Prediction: 1


In [11]:
import os

# Path to the generated C header file
model_file_path = 'my_model.h'

# Check if the file exists and print the first few lines
if os.path.exists(model_file_path):
    print("First few lines of my_model.h:\n")
    with open(model_file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= 20:
                break
            print(line.rstrip())
else:
    print(f"Error: Model file '{model_file_path}' not found.")


First few lines of my_model.h:




    // !!! This file is generated using emlearn !!!

    #include <stdint.h>


static inline int32_t my_model_tree_0(const int16_t *features, int32_t features_length) {
          if (features[0] < 127) {
              return 0;
          } else {
              if (features[1] < 127) {
                  return 0;
              } else {
                  return 1;
              }
          }
        }



##### Porting to Microcontroller - Arduino

- Try this code on Arduino by following the steps mentioned [here](https://emlearn.readthedocs.io/en/latest/getting_started_arduino.html).

### Part B:  TFLite Micro for Microcontrollers

##### Step 1 :  Train a model

Use Fibonacci dataset for supervised training to train a fully connected network. Save trained model in Tflite (Flatbuffer) format

In [0]:
import warnings
warnings.filterwarnings("ignore")

from numpy import array
from keras.models import Sequential
from keras.layers import Dense, Input
import tensorflow as tf


# split a univariate sequence into samples
def split_sequence(sequence, n_steps_in, n_steps_out):
	X, y = list(), list()
	for i in range(len(sequence)):
		# find the end of this pattern
		end_index = i + n_steps_in
		out_end_index = end_index + n_steps_out
		# check if we are beyond the sequence
		if out_end_index > len(sequence):
			break
		# gather input and output parts of the pattern
		seq_x, seq_y = sequence[i:end_index], sequence[end_index:out_end_index]
		X.append(seq_x)
		y.append(seq_y)
	return array(X), array(y)

#experimental Fibonaaci numbers
raw_seq = [0,1,1,2,3,5,8,13,21,34,55,89,144]

# choose a number of time steps for input and output
n_steps_in, n_steps_out = 3,2

# split into samples
X, y = split_sequence(raw_seq, n_steps_in, n_steps_out)

#print (X,y)
#print (X.shape, y.shape)

# define model
model = Sequential()
model.add(Input(shape=(n_steps_in,)))
model.add(Dense(25, activation='relu'))
model.add(Dense(25, activation='relu'))
model.add(Dense(n_steps_out))
model.compile(optimizer='adam', loss='mse')

# model architecture
model.summary()

# fit model
model.fit(X, y, epochs=200, verbose=0)

# Convert the model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TF Lite model.
with tf.io.gfile.GFile('fibonacci.tflite', 'wb') as f:
  f.write(tflite_model)

In [0]:
# Convert to a C source file, i.e, a TensorFlow Lite for Microcontrollers model
!xxd -i fibonacci.tflite > fibo_model.h
!sed -i 's/unsigned char /const unsigned char /g' fibo_model.h
!sed -i 's/const/alignas(8) const/g' fibo_model.h

The last two lines that use the *sed* command are needed to make sure that the model resides in the program memory (Flash) and fits in the 8-bytes boundary. You can print out the model content by **!cat fibo_model.h**

In [0]:
!cat fibo_model.h

##### Step 2. Generate Model with Quantization

- We now have an acceptably accurate model.
- Use the [TensorFlow Lite Converter](https://www.tensorflow.org/lite/convert) to convert the model into a space-efficient format for memory-constrained devices.
- Since the model will be deployed on a microcontroller, it should be as small as possible.
- Quantization is a technique for reducing model size.
- It lowers the precision of the model’s weights, and sometimes the activations too.
- This saves memory and often has little impact on accuracy.
- Quantized models can also run faster because the computations are simpler.
- In the next cell, the model will be converted twice: once with quantization and once without.

In [0]:
import numpy as np
# Convert the model to the TensorFlow Lite format with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Indicate that we want to perform the default optimizations,
# which includes quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Define a generator function that provides our test data's x values
# as a representative dataset, and tell the converter to use it
def representative_dataset_generator():
  for value in X:
    # Each scalar value must be inside of a 2D array that is wrapped in a list
    yield [np.array(value, dtype=np.float32, ndmin=2)]
converter.representative_dataset = representative_dataset_generator
# Convert the model

tflite_model = converter.convert()

# Save the model to disk
open("fibonacci_quantized.tflite", "wb").write(tflite_model)


Download the fibo_model.h and the fibonacci_quantized.tflite file from the Output tab. 

##### **Step 3:**

Generate a TensorFlow Lite for Microcontrollers Model
Convert the TensorFlow Lite quantized model into a C source file that can be loaded by TensorFlow Lite for Microcontrollers.

In [0]:
# Convert to a C source file, i.e, a TensorFlow Lite for Microcontrollers model
!xxd -i fibonacci_quantized.tflite > fibo_quan_model.h
!sed -i 's/unsigned char /const unsigned char /g' fibo_quan_model.h
!sed -i 's/const/alignas(8) const/g' fibo_quan_model.h

#### **Step 4:**

Download and Deploy to a Microcontroller. fibonacci_quantized.cc, (5024 byte file) contains information about the model architecture and values of 802 parameters.


##### **Demo of model execution on microcontroller**

- The instructor will demonstrate Fibonacci model on a Cortex M-4 based microcontroller board. The TFLite Micro (TFLM) C++ code uses the byte array formatted model schema created earlier.
- Our Fibonacci model is a good example of a  model which does not require feature normalization during training and inference. Most models do require feature normalization and quantization.


##### **Exercise:**

Use dataset from exercise-5 which has the following 5 features,
- First three columns - Acceleration data of X,Y,Z axis
- Last two columns - Temperature and Humidity

This data could be of a fragile container being shipped in a controlled temperature enviroment. The container could be in one of 4 states in terms of position/temperature/humidity.

- We provide you with a sample dataset to get started.

Scale the features in the dataset and then create a fully connected model to train on this dataset.

Save the trained model in TFlite format.

The purpose of the exercise is to transform logged data from sensors and make it suitable for supervised AI model training.

In [0]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/data_acq_1.txt

##### Solution to Exercise: 

<details>
    <summary> Click here to view our answer </summary>

    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler

    try:
        df = pd.read_csv('data_acq_1.txt')
        print("CSV file read successfully into a pandas DataFrame.")
        print(df.head())

        # Convert to a NumPy array and assign to Train_X
        Train_X = df.to_numpy()
        print("\nDataFrame converted to NumPy array and assigned to Train_X.")
        print(Train_X[:5]) # Print first 5 rows of the NumPy array

        # Normalize the data in Train_X
        scaler = StandardScaler()
        Train_X = scaler.fit_transform(Train_X)
        print("\nTrain_X data normalized using StandardScaler.")
        print(Train_X[:5]) # Print first 5 rows of the normalized NumPy array


        # Create a NumPy array for labels with random values from 0 to 3
        Train_Y = np.random.randint(0, 4, size=Train_X.shape[0])
        print("\nCreated a NumPy array for labels (Train_Y) with random values from 0 to 3.")
        print(Train_Y[:5]) # Print first 5 elements of the Train_Y array

    except FileNotFoundError:
        print("Error: 'data_acq_1.txt' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    # Print 5 rows of the scaled data

    # Create a DataFrame from the first 5 rows of Train_X and the first 5 labels from Train_Y
    X = pd.DataFrame(Train_X[:5])
    y = Train_Y[:5]

    # Create a DataFrame for printing, including the labels
    df_subset = pd.DataFrame(X)
    df_subset['label'] = y

    # Print the DataFrame
    print(df_subset)

    # Perform training and save model in tflite format

    from numpy import array
    from keras.models import Sequential
    from keras.layers import Dense, Input
    import tensorflow as tf

    # define model
    model = Sequential()
    model.add(Input(shape=(5,)))
    model.add(Dense(25, activation='relu'))
    model.add(Dense(25, activation='relu'))
    model.add(Dense(4, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

    # model architecture
    model.summary()

    # fit model
    model.fit(X, y, epochs=200, verbose=0)

    # Convert the model to TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()

    # Save the TF Lite model.
    with tf.io.gfile.GFile('vibration_temp.tflite', 'wb') as f:
      f.write(tflite_model)
    
    
</details>